# SIMAT 2026 · Notebook 2
## Covarianza, correlación, multicolinealidad e influencia

**Archivo local:** `contaminantes_2026.csv` · **Estación de regresión:** MER (Merced).

Este notebook continúa la metodología del ejercicio anterior: una explicación en Markdown antes de cada celda de código, tablas identificadas y conclusiones calculadas a partir de los datos.

| Inciso | Actividad | Muestra |
|---|---|---|
| a | Una fila por estación-hora y una columna por contaminante | Todas las estaciones de 2026 |
| b | Covarianza y correlación de CO con NOX | Pares completos, total y MER |
| c | Eigenvalores de la correlación, con y sin faltantes | Todas las estaciones; eliminación por pares frente a casos completos |
| d | PM2.5 contra los demás: $R^2$ y VIF; repetir solo con gases | MER, misma muestra para ambos modelos |
| e | Matriz $H$, idempotencia, traza y leverage sin construir $H$ | Una semana de MER para $H$; todo el periodo de MER para leverage |
| f | Distancia de Cook, diez más influyentes y reajuste | Ambos modelos de MER |

**Cómo ejecutar:** coloca el CSV junto al notebook (o en `/content` en Colab), revisa la ruta en la siguiente celda y ejecuta todo en orden. No se descarga ni se sube ningún archivo. Se utiliza exclusivamente el año 2026 presente en el CSV.

**Estado de los resultados:** este archivo contiene el análisis completo y genera sus tablas, gráficas e interpretaciones al ejecutarse. No contiene resultados numéricos atribuidos a SIMAT sin haber leído el CSV. Las pruebas de funcionamiento realizadas por separado con datos sintéticos no forman parte de este notebook.


In [ ]:
# Si falta una dependencia, ejecuta en otra celda:
# %pip install numpy pandas matplotlib seaborn statsmodels

from pathlib import Path
import csv
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display, Markdown

RUTA = Path("contaminantes_2026.csv")
ESTACION = "MER"
ANIO = 2026
CONTAMINANTES = ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PM2.5", "PMCO", "SO2"]
GASES = ["CO", "NO", "NO2", "NOX", "O3", "SO2"]
OBJETIVO = "PM2.5"
# None: primera ventana de siete días con suficientes casos completos.
# Para fijarla, escribe por ejemplo "2026-01-01".
INICIO_SEMANA = None

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)


## Preparación · lectura local y control de calidad

Localizamos el encabezado real después de los metadatos y convertimos `date` y `valor`. Las fechas se conservan como aparecen en SIMAT, sin convertirlas a UTC. No imputamos faltantes ni cambiamos unidades. Los códigos de `unit` son identificadores: **no son factores de conversión**.

Eliminamos solo registros duplicados idénticos. Si una estación-hora-contaminante tiene valores contradictorios, detenemos el análisis para no elegir uno arbitrariamente. Se informa cuántas fechas o mediciones no se pudieron interpretar. Los negativos se reportan para revisión; no se consideran automáticamente códigos de ausencia sin consultar el diccionario del archivo.


In [ ]:
if not RUTA.is_file():
    raise FileNotFoundError(
        f"No se encontró {RUTA.resolve()}. Coloca el CSV junto al notebook "
        "o modifica RUTA (por ejemplo, /content/contaminantes_2026.csv)."
    )

for encoding in ("utf-8-sig", "cp1252"):
    try:
        texto = RUTA.read_text(encoding=encoding)
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("No fue posible reconocer la codificación del CSV.")

lineas = texto.splitlines()
encabezado = next((i for i, linea in enumerate(lineas)
                   if [x.strip().lower() for x in next(csv.reader([linea]))][:3]
                   == ["date", "id_station", "id_parameter"]), None)
if encabezado is None:
    raise ValueError("No se encontró el encabezado date,id_station,id_parameter.")

print("Metadatos del archivo:")
print("\n".join(lineas[:encabezado]))
df = pd.read_csv(RUTA, skiprows=encabezado, encoding=encoding, low_memory=False)
df.columns = df.columns.str.strip()
requeridas = {"date", "id_station", "id_parameter", "valor", "unit"}
if not requeridas.issubset(df.columns):
    raise ValueError(f"Faltan columnas: {requeridas - set(df.columns)}")

valor_original = df["valor"].copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
print("Fechas inválidas:", df["date"].isna().sum())
print("Valores no numéricos convertidos en faltantes:",
      (valor_original.notna() & df["valor"].isna()).sum())
print("Valores infinitos convertidos en faltantes:", np.isinf(df["valor"]).sum())
df["valor"] = df["valor"].replace([np.inf, -np.inf], np.nan)
for columna in ["id_station", "id_parameter"]:
    df[columna] = df[columna].astype("string").str.strip().str.upper()
df = df.loc[(df["date"].dt.year == ANIO)
            & df["id_parameter"].isin(CONTAMINANTES)].copy()
if df.empty:
    raise ValueError("No hay registros de los contaminantes solicitados en 2026.")
if df["id_station"].isna().any() or df["id_station"].eq("").any():
    raise ValueError("Hay registros sin identificador de estación; revisa el CSV.")

unidades = df.groupby("id_parameter")["unit"].agg(
    lambda s: sorted(s.dropna().astype(str).unique().tolist()))
display(unidades.rename("códigos de unidad registrados").to_frame())
if unidades.map(len).gt(1).any():
    raise ValueError("Un contaminante usa distintos códigos de unidad; homogeneízalos antes de continuar.")
print("Mediciones negativas (revisar si son válidas):", df["valor"].lt(0).sum())

claves = ["date", "id_station", "id_parameter"]
n_antes = len(df)
df = df.drop_duplicates(subset=claves + ["valor"])
print("Duplicados idénticos eliminados:", n_antes - len(df))
conflictos = df.duplicated(claves, keep=False)
if conflictos.any():
    display(df.loc[conflictos].sort_values(claves).head(20))
    raise ValueError("Hay valores contradictorios para una misma estación-hora-contaminante.")


## a) Tabla estación-hora × contaminante

Usamos `pivot` porque ya comprobamos que cada combinación es única. Conservamos los `NaN`: una medición ausente no equivale a una concentración cero. A diferencia de una agregación diaria, aquí la fecha conserva la hora completa. La tabla representa las combinaciones presentes en el archivo; no inventa horas que no aparecen.


In [ ]:
tabla = (df.pivot(index=["date", "id_station"], columns="id_parameter", values="valor")
         .reindex(columns=CONTAMINANTES).sort_index())
tabla.columns.name = None
tabla_estacion_hora = tabla.reset_index()
display(tabla_estacion_hora.head(12))
print(f"Tabla: {len(tabla):,} filas estación-hora × {len(CONTAMINANTES)} contaminantes.")
print("Periodo:", tabla.index.get_level_values("date").min(),
      "a", tabla.index.get_level_values("date").max())
display(pd.DataFrame({
    "datos_disponibles": tabla.count(),
    "faltantes": tabla.isna().sum(),
    "porcentaje_faltante": 100 * tabla.isna().mean()
}).round(2))
if ESTACION not in tabla.index.get_level_values("id_station"):
    raise ValueError(f"La estación {ESTACION} no aparece en este archivo.")
mer = tabla.xs(ESTACION, level="id_station").sort_index()


## b) Covarianza y correlación de CO con NOX

Calculamos ambas medidas con **los mismos pares completos** y usando la covarianza muestral (`ddof=1`):

$$s_{CO,NOX}=\frac{\sum_{i=1}^n(CO_i-\overline{CO})(NOX_i-\overline{NOX})}{n-1},
\qquad r_{CO,NOX}=\frac{s_{CO,NOX}}{s_{CO}s_{NOX}}.$$

La covarianza conserva unidades de CO × NOX y cambia al cambiar la escala. La correlación es adimensional y está entre −1 y 1. Por tanto, **son medidas distintas**, aunque ambas describen asociación lineal y, si están definidas, tienen el mismo signo. Una covarianza mayor numéricamente no representa por sí misma una relación más fuerte.

Mostramos la comparación global y la de MER. La asociación global mezcla diferencias entre estaciones y diferencias en el tiempo; no representa automáticamente la asociación dentro de cada estación ni una relación causal.


In [ ]:
def resumen_co_nox(base, nombre):
    pares = base[["CO", "NOX"]].dropna()
    if len(pares) < 3 or (pares.std(ddof=1) == 0).any():
        raise ValueError(f"{nombre}: faltan pares suficientes con variación.")
    cov = pares.cov().loc["CO", "NOX"]
    corr = pares.corr().loc["CO", "NOX"]
    verificacion = cov / (pares["CO"].std(ddof=1) * pares["NOX"].std(ddof=1))
    assert np.isclose(corr, verificacion)
    return {"muestra": nombre, "n_pares": len(pares), "covarianza": cov,
            "correlacion": corr, "cov/(s_CO*s_NOX)": verificacion}

resumen_b = pd.DataFrame([
    resumen_co_nox(tabla, "Todas las estaciones"),
    resumen_co_nox(mer, ESTACION)
]).set_index("muestra")
display(resumen_b.round(6))
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(mer["CO"], mer["NOX"], s=14, alpha=0.35)
ax.set(xlabel="CO (unidad original)", ylabel="NOX (unidad original)",
       title=f"{ESTACION}: asociación entre CO y NOX")
plt.show()


## c) Eigenvalores de la matriz de correlación, con y sin faltantes

Comparamos dos tratamientos sobre **las mismas variables**:

1. **Con faltantes: eliminación por pares.** Cada correlación utiliza las filas disponibles para ese par. También calculamos el número de pares usados en cada entrada.
2. **Sin faltantes: casos completos.** Todas las correlaciones utilizan exactamente las mismas filas después de `dropna()`.

Una matriz de correlación construida sobre una muestra común es semidefinida positiva: sus eigenvalores son no negativos, salvo error numérico. En cambio, una matriz calculada por pares **puede** tener eigenvalores negativos, porque sus entradas provienen de muestras distintas. No es obligatorio que ocurra. La suma de eigenvalores debe coincidir con la traza, que equivale al número de variables cuando la diagonal es uno.

Los eigenvalores próximos a cero indican relaciones lineales casi redundantes. No reemplazamos correlaciones indefinidas por cero ni rellenamos los datos artificialmente.


In [ ]:
variables_c = [v for v in CONTAMINANTES
               if tabla[v].notna().sum() >= 3 and tabla[v].nunique() > 1]
excluidas_c = sorted(set(CONTAMINANTES) - set(variables_c))
print("Variables excluidas por ausencia de datos o falta de variación:", excluidas_c)
if len(variables_c) < 2:
    raise ValueError("Se necesitan al menos dos variables con variación.")

base_c = tabla[variables_c]
R_pares = base_c.corr(min_periods=3)
validos = base_c.notna().astype(np.int64)
n_pares = validos.T @ validos
completos_c = base_c.dropna()
if len(completos_c) < 3 or (completos_c.std() == 0).any():
    raise ValueError("No hay casos completos suficientes con variación para comparar eigenvalores.")
R_completos = completos_c.corr()
if R_pares.isna().any().any():
    display(R_pares)
    raise ValueError("Alguna correlación por pares no está definida. Revisa cobertura y variación antes de calcular eigenvalores.")

print("Número de observaciones usadas por cada par:")
display(n_pares)
print(f"Casos completos comunes: {len(completos_c):,} de {len(base_c):,}.")
print("Porcentaje de filas eliminado:", round(100 * (1 - len(completos_c)/len(base_c)), 2))

# eigvalsh aprovecha que las matrices son simétricas y devuelve valores reales.
eig_pares = np.linalg.eigvalsh(R_pares.to_numpy())
eig_completos = np.linalg.eigvalsh(R_completos.to_numpy())
eigenvalores = pd.DataFrame({"orden": np.arange(1, len(variables_c)+1),
                            "por_pares": eig_pares, "casos_completos": eig_completos})
display(eigenvalores)
TOL_EIG = 1e-8
resumen_c = pd.DataFrame([
    {"método": nombre, "lambda_min": valores.min(),
     "suma_eigenvalores": valores.sum(), "traza": np.trace(matriz),
     "negativos_menores_a_-1e-8": int((valores < -TOL_EIG).sum())}
    for nombre, valores, matriz in [
        ("Por pares", eig_pares, R_pares),
        ("Casos completos", eig_completos, R_completos)]
]).set_index("método")
display(resumen_c)
assert np.allclose(eig_pares.sum(), np.trace(R_pares))
assert eig_completos.min() >= -TOL_EIG

fig, ax = plt.subplots(1, 3, figsize=(17, 4), constrained_layout=True)
sns.heatmap(R_pares, vmin=-1, vmax=1, cmap="coolwarm", ax=ax[0])
ax[0].set_title("Correlación por pares")
sns.heatmap(R_completos, vmin=-1, vmax=1, cmap="coolwarm", ax=ax[1])
ax[1].set_title("Correlación: casos completos")
ax[2].plot(eigenvalores["orden"], eig_pares, "o-", label="Por pares")
ax[2].plot(eigenvalores["orden"], eig_completos, "s--", label="Completos")
ax[2].axhline(0, color="black", linewidth=1)
ax[2].set(xlabel="Orden del eigenvalor (ascendente)", ylabel="Eigenvalor")
ax[2].legend()
plt.show()

mensaje_c = ("La matriz por pares tiene eigenvalores negativos relevantes y no es semidefinida positiva."
             if (eig_pares < -TOL_EIG).any()
             else "La matriz por pares no presenta eigenvalores negativos relevantes en este archivo; eso no está garantizado en otras muestras.")
display(Markdown(f"**Interpretación:** {mensaje_c} La matriz de casos completos utiliza {len(completos_c):,} filas. "
                 "Las diferencias entre matrices también reflejan la selección de observaciones; eliminar faltantes no garantiza ausencia de sesgo."))


## d) PM2.5 contra los demás contaminantes; después solo gases

Ajustamos dos modelos con intercepto en MER:

- **Todos:** `PM2.5 ~ CO + NO + NO2 + NOX + O3 + PM10 + PMCO + SO2`.
- **Gases:** `PM2.5 ~ CO + NO + NO2 + NOX + O3 + SO2`.

Ambos se ajustan sobre las **mismas horas completas de las nueve variables**, para que la comparación de $R^2$ no se deba a cambiar la muestra. Reportamos también cuántas horas podrían usarse si ajustáramos solo gases por separado.

Para cubrir las dos interpretaciones de “$R^2$ y VIF”, calculamos:

1. El $R^2$ de cada regresión de PM2.5 y el VIF de **cada predictor** mediante regresiones auxiliares.
2. $1/(1-R^2)$ de la regresión de PM2.5: es el **VIF de PM2.5 si fuera una variable explicativa dentro del conjunto**, no un VIF habitual de la respuesta en su propio modelo.

Se verifica la cercanía de `PM10 = PM2.5 + PMCO` y `NOX = NO + NO2`. Un ajuste elevado por una identidad entre mediciones no equivale a capacidad de pronóstico. Aquí se explica una medición simultánea, no una concentración futura.


In [ ]:
PRED_TODOS = [v for v in CONTAMINANTES if v != OBJETIVO]
muestra = mer[CONTAMINANTES].dropna().copy()
n_gases_disponibles = len(mer[[OBJETIVO] + GASES].dropna())
print(f"MER: {len(mer):,} horas registradas; {len(muestra):,} completas para comparar modelos.")
print(f"Horas completas para PM2.5 + gases solamente: {n_gases_disponibles:,}.")
if len(muestra) <= len(PRED_TODOS) + 2:
    raise ValueError("MER no tiene suficientes casos completos para estos modelos.")
if (muestra.std() == 0).any():
    raise ValueError("Hay una variable constante en la muestra; revisa la cobertura.")

identidades = pd.DataFrame({
    "NOX - NO - NO2": muestra["NOX"] - muestra["NO"] - muestra["NO2"],
    "PM10 - PM2.5 - PMCO": muestra["PM10"] - muestra["PM2.5"] - muestra["PMCO"]
})
display(pd.DataFrame({"porcentaje_igualdad_exacta": 100*identidades.eq(0).mean(),
                      "mediana_error_absoluto": identidades.abs().median(),
                      "max_error_absoluto": identidades.abs().max()}))

# SVD: base ortonormal del espacio columna; evita invertir X'X.
def base_proyeccion(X):
    A = np.asarray(X, dtype=float)
    U, s, _ = np.linalg.svd(A, full_matrices=False)
    tolerancia = max(A.shape) * np.finfo(float).eps * s[0]
    rango = int(np.sum(s > tolerancia))
    return U[:, :rango], rango

def ajustar(base, predictores):
    X = sm.add_constant(base[predictores], has_constant="add")
    return sm.OLS(base[OBJETIVO], X).fit()

def calcular_vif(base, variables):
    filas = []
    for variable in variables:
        restantes = [v for v in variables if v != variable]
        auxiliar = sm.OLS(base[variable], sm.add_constant(base[restantes], has_constant="add")).fit()
        r2 = auxiliar.rsquared
        vif = np.inf if 1-r2 <= 1e-12 else 1/(1-r2)
        filas.append({"variable": variable, "R2_auxiliar": r2, "VIF": vif})
    return pd.DataFrame(filas).set_index("variable")

predictores_modelos = {"Todos": PRED_TODOS, "Gases": GASES}
modelos = {nombre: ajustar(muestra, pred) for nombre, pred in predictores_modelos.items()}
resumen_d = []
for nombre, modelo in modelos.items():
    rango = base_proyeccion(modelo.model.exog)[1]
    resumen_d.append({"modelo": nombre, "n": int(modelo.nobs),
                      "columnas_X_incl_intercepto": modelo.model.exog.shape[1], "rango_X": rango,
                      "R2": modelo.rsquared, "R2_ajustado": modelo.rsquared_adj,
                      "RMSE_ajuste": np.sqrt(np.mean(modelo.resid**2)),
                      "VIF_PM2.5_auxiliar": np.inf if 1-modelo.rsquared <= 1e-12 else 1/(1-modelo.rsquared)})
resumen_d = pd.DataFrame(resumen_d).set_index("modelo")
display(resumen_d)
vifs = {nombre: calcular_vif(muestra, pred) for nombre, pred in predictores_modelos.items()}
display(pd.concat(vifs, names=["modelo", "variable"]))
display(pd.concat({nombre: modelo.params for nombre, modelo in modelos.items()}, axis=1))
assert modelos["Todos"].rsquared + 1e-10 >= modelos["Gases"].rsquared


## d) Lectura de la comparación

El modelo con todos los contaminantes contiene al de gases, por lo que su $R^2$ de entrenamiento no puede ser menor en la misma muestra. Esto no demuestra que pronostique mejor. La identidad de las partículas puede producir un ajuste casi mecánico; para valorar utilidad predictiva se necesitaría una prueba temporal fuera de muestra y variables disponibles al momento de predecir.

Un VIF elevado señala dificultad para separar coeficientes individuales. Si aparece `inf`, hay redundancia exacta o numéricamente indistinguible de ella. Si el rango de $X$ es menor que su número de columnas, los coeficientes individuales no son únicos; `statsmodels` devuelve una solución mediante pseudoinversa.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, (nombre, modelo) in zip(axes, modelos.items()):
    ax.scatter(muestra[OBJETIVO], modelo.fittedvalues, s=14, alpha=0.35)
    limite_min = min(muestra[OBJETIVO].min(), modelo.fittedvalues.min())
    limite_max = max(muestra[OBJETIVO].max(), modelo.fittedvalues.max())
    ax.plot([limite_min, limite_max], [limite_min, limite_max], "r--")
    ax.set(title=f"{nombre}: R² = {modelo.rsquared:.4f}",
           xlabel="PM2.5 observado", ylabel="PM2.5 ajustado")
plt.show()
display(Markdown(
    f"**Resultado d:** con todos los contaminantes, $R^2={modelos['Todos'].rsquared:.4f}$; "
    f"con gases, $R^2={modelos['Gases'].rsquared:.4f}$. "
    f"Ambos usan las mismas **{len(muestra):,} horas**. "
    "La tabla de identidades permite evaluar cuánto del ajuste puede explicarse por redundancia entre mediciones."
))


## e) Matriz de proyección $H$ para una semana de MER

Para un modelo OLS, $\widehat y=Hy$. Si $X$ tiene rango completo,

$$H=X(X^TX)^{-1}X^T.$$

Usaremos la forma numéricamente más estable $H=UU^T$, donde las columnas de $U$ son una base ortonormal del espacio columna de $X$, obtenida por SVD. Esto también cubre diseños deficientes de rango.

Comprobamos $H^2=H$ (idempotencia), $H^T=H$ (simetría) y:

$$\operatorname{tr}(H)=\operatorname{rango}(X).$$

Si $p$ cuenta las columnas de $X$ **incluido el intercepto** y hay rango completo, entonces $\operatorname{tr}(H)=p$. Si hay redundancia exacta, la traza es el rango efectivo y no el número nominal de columnas. No forzamos una igualdad falsa.

La semana es una ventana de siete días calendario, con hasta 168 horas. Las horas incompletas se excluyen; no sustituimos “una semana” por las primeras 168 filas disponibles. Comprobamos las propiedades para ambos modelos.


In [ ]:
if INICIO_SEMANA is None:
    candidatos = pd.date_range(muestra.index.min().normalize(),
                               muestra.index.max().normalize(), freq="D")
else:
    candidatos = [pd.Timestamp(INICIO_SEMANA).normalize()]

semana = None
for inicio in candidatos:
    fin = inicio + pd.Timedelta(days=7)
    candidata = muestra.loc[(muestra.index >= inicio) & (muestra.index < fin)]
    if len(candidata) > len(PRED_TODOS) + 2:
        semana = candidata.copy()
        break
if semana is None:
    raise ValueError("No hay suficientes horas completas en la semana seleccionada.")
print(f"Semana: [{inicio}, {fin}) — extremo final excluido.")
print(f"Horas completas: {len(semana)} de hasta 168.")

matrices_H = {}
resumen_e = []
for nombre, pred in predictores_modelos.items():
    X_semana = sm.add_constant(semana[pred], has_constant="add")
    U, rango = base_proyeccion(X_semana)
    H = U @ U.T
    matrices_H[nombre] = H
    p = X_semana.shape[1]
    resumen_e.append({"modelo": nombre, "n_semana": len(semana), "p_columnas": p,
                      "rango_X": rango, "traza_H": np.trace(H),
                      "max_abs_H2_menos_H": np.max(np.abs(H @ H-H)),
                      "max_abs_HT_menos_H": np.max(np.abs(H.T-H)),
                      "traza_igual_p": np.isclose(np.trace(H), p)})
    assert np.allclose(H @ H, H, atol=1e-9)
    assert np.allclose(H.T, H, atol=1e-9)
    assert np.isclose(np.trace(H), rango)
    assert np.allclose(np.diag(H), (U**2).sum(axis=1))
resumen_e = pd.DataFrame(resumen_e).set_index("modelo")
display(resumen_e)


## e) Leverage de todas las horas de MER sin construir $H$

El leverage $h_{ii}$ mide lo inusual que es la combinación de predictores de una observación respecto al resto. Depende de $X$, no de la respuesta observada.

Con la misma base ortonormal:

$$h_{ii}=\sum_{j=1}^{r}U_{ij}^{2},\qquad \sum_i h_{ii}=r,\qquad 0\le h_{ii}\le1,$$

donde $r$ es el rango. Calculamos únicamente un vector de longitud $n$: **no construimos una matriz $n\times n$ para todo el periodo**. El umbral $2r/n$ se muestra como referencia descriptiva, no como prueba de hipótesis ni criterio automático de exclusión. “Todos” se refiere a todas las horas completas de MER de la muestra de regresión.


In [ ]:
leverages = {}
resumen_leverage = []
for nombre, modelo in modelos.items():
    U, rango = base_proyeccion(modelo.model.exog)
    h = (U**2).sum(axis=1)
    leverages[nombre] = pd.Series(h, index=muestra.index, name="leverage")
    assert np.all(h >= -1e-10) and np.all(h <= 1+1e-10)
    assert np.isclose(h.sum(), rango)
    resumen_leverage.append({"modelo": nombre, "n": len(h), "rango": rango,
                             "suma_h": h.sum(), "promedio_h": h.mean(),
                             "r_sobre_n": rango/len(h), "max_h": h.max(),
                             "umbral_2r_n": 2*rango/len(h),
                             "n_sobre_umbral": int((h > 2*rango/len(h)).sum())})
display(pd.DataFrame(resumen_leverage).set_index("modelo"))
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
for ax, (nombre, h) in zip(axes, leverages.items()):
    r = base_proyeccion(modelos[nombre].model.exog)[1]
    ax.scatter(h.index, h.values, s=12, alpha=0.6)
    ax.axhline(2*r/len(h), color="crimson", linestyle="--", label="2r/n")
    ax.set(title=f"{nombre}: leverage en todo el periodo de MER", ylabel="hᵢᵢ", xlabel="Fecha")
    ax.legend()
plt.show()


## f) Distancia de Cook: ¿influencia y leverage son lo mismo?

La distancia de Cook combina el tamaño del residuo con el leverage:

$$D_i=\frac{e_i^2}{r\,\widehat\sigma^2}\frac{h_{ii}}{(1-h_{ii})^2},
\qquad \widehat\sigma^2=\frac{\sum_i e_i^2}{n-r}.$$

En rango completo, $r=p$. Un punto puede tener mucho leverage y poca influencia si está bien ajustado; también puede ser influyente por un residuo grande. Por tanto, los diez mayores valores de Cook y los diez mayores leverages **no tienen por qué coincidir**.

Para cada modelo calculamos ambos rankings con toda la muestra de MER, mostramos la intersección y reajustamos después de excluir **una sola vez sus diez mayores valores de Cook**. Los empates se resuelven conservando el orden cronológico. $4/n$ es una referencia exploratoria, no una regla que obligue a eliminar datos. Si el ajuste es numéricamente perfecto, Cook no es informativo porque la varianza residual es cero; ese caso se reporta explícitamente.


In [ ]:
influencia = {}
top_cook = {}
top_leverage = {}
comparacion_ranking = []

for nombre, modelo in modelos.items():
    e = np.asarray(modelo.resid)
    h = leverages[nombre].to_numpy()
    n = len(e)
    r = base_proyeccion(modelo.model.exog)[1]
    sse = float(e @ e)
    sst = float(np.sum((muestra[OBJETIVO] - muestra[OBJETIVO].mean())**2))
    if sse <= 1e-12 * max(sst, 1.0):
        warnings.warn(f"{nombre}: ajuste numéricamente perfecto; Cook no es interpretable.")
        continue
    if np.any(1-h <= 1e-10):
        raise ValueError(f"{nombre}: hay leverage igual a uno; la eliminación requiere tratamiento específico.")
    mse = sse/(n-r)
    cook = (e**2/(r*mse)) * h/((1-h)**2)
    t = pd.DataFrame({"PM2.5": muestra[OBJETIVO], "ajustado": modelo.fittedvalues,
                      "residuo": e, "leverage": h, "Cook": cook}, index=muestra.index)
    t["ranking_Cook"] = t["Cook"].rank(ascending=False, method="first").astype(int)
    t["ranking_leverage"] = t["leverage"].rank(ascending=False, method="first").astype(int)
    influencia[nombre] = t
    top_cook[nombre] = t.sort_values("Cook", ascending=False, kind="stable").head(10).index
    top_leverage[nombre] = t.sort_values("leverage", ascending=False, kind="stable").head(10).index
    comunes = top_cook[nombre].intersection(top_leverage[nombre])
    comparacion_ranking.append({"modelo": nombre, "coincidencias_entre_top10": len(comunes),
                               "mismo_conjunto": set(top_cook[nombre]) == set(top_leverage[nombre]),
                               "n_Cook_mayor_4_n": int((cook > 4/n).sum())})
    display(Markdown(f"### {nombre}: diez observaciones de mayor distancia de Cook"))
    display(t.loc[top_cook[nombre]])
    display(Markdown(f"**Diez mayores leverages — {nombre}**"))
    display(t.loc[top_leverage[nombre], ["leverage", "Cook", "ranking_Cook"]])

if comparacion_ranking:
    display(pd.DataFrame(comparacion_ranking).set_index("modelo"))
    fig, axes = plt.subplots(1, len(influencia), figsize=(7*len(influencia), 4), squeeze=False)
    for ax, (nombre, t) in zip(axes.ravel(), influencia.items()):
        ax.scatter(t["leverage"], t["Cook"], s=16, alpha=0.4)
        destacados = t.loc[top_cook[nombre]]
        ax.scatter(destacados["leverage"], destacados["Cook"], color="crimson", label="Top 10 Cook")
        ax.axhline(4/len(t), color="grey", linestyle="--", label="4/n")
        ax.set(xlabel="Leverage", ylabel="Distancia de Cook", title=nombre)
        ax.legend()
    plt.tight_layout()
    plt.show()


## f) Reajuste sin los diez más influyentes

Reajustamos cada modelo en sus propias filas retenidas. Reportamos el cambio de coeficientes y dos comparaciones:

- **Ajuste en su muestra:** $R^2$ original y $R^2$ del reajuste. Como cambia la muestra, no basta comparar estos dos valores para afirmar una mejora.
- **Mismas filas retenidas:** evaluamos el modelo original y el reajustado sobre exactamente esas filas. Es una comparación descriptiva dentro de muestra, no una validación predictiva independiente.

La exclusión responde al ejercicio y constituye un análisis de sensibilidad. No demuestra que las diez observaciones sean incorrectas. Si el rango cambia al retirarlas, también cambia la identificabilidad; si hay colinealidad extrema, los cambios en coeficientes deben interpretarse con especial cautela.


In [ ]:
reajustes = {}
filas_reajuste = []
cambios_coeficientes = {}

def metricas_en(y, predicho):
    y = np.asarray(y, dtype=float)
    predicho = np.asarray(predicho, dtype=float)
    sse = np.sum((y-predicho)**2)
    sst = np.sum((y-y.mean())**2)
    return (1-sse/sst if sst > 0 else np.nan), np.sqrt(sse/len(y))

for nombre in influencia:
    pred = predictores_modelos[nombre]
    retenidas = muestra.drop(index=top_cook[nombre])
    original = modelos[nombre]
    nuevo = ajustar(retenidas, pred)
    reajustes[nombre] = nuevo
    X_retenidas = sm.add_constant(retenidas[pred], has_constant="add")
    r2_antes, rmse_antes = metricas_en(retenidas[OBJETIVO], original.predict(X_retenidas))
    r2_despues, rmse_despues = metricas_en(retenidas[OBJETIVO], nuevo.predict(X_retenidas))
    filas_reajuste.append({"modelo": nombre, "n_original": len(muestra), "n_reajuste": len(retenidas),
                          "rango_original": base_proyeccion(original.model.exog)[1],
                          "rango_reajuste": base_proyeccion(nuevo.model.exog)[1],
                          "R2_original_muestra_original": original.rsquared,
                          "R2_original_en_retenidas": r2_antes,
                          "R2_reajuste_en_retenidas": r2_despues,
                          "RMSE_original_en_retenidas": rmse_antes,
                          "RMSE_reajuste_en_retenidas": rmse_despues})
    cambios_coeficientes[nombre] = pd.DataFrame({
        "coef_original": original.params, "coef_reajuste": nuevo.params,
        "cambio_absoluto": nuevo.params-original.params})

resumen_f = pd.DataFrame(filas_reajuste)
if not resumen_f.empty:
    resumen_f = resumen_f.set_index("modelo")
    display(resumen_f)
    display(pd.concat(cambios_coeficientes, names=["modelo", "coeficiente"]))
    display(Markdown("**VIF después del reajuste:** se recalcula para comprobar si la redundancia persiste."))
    display(pd.concat({nombre: calcular_vif(muestra.drop(index=top_cook[nombre]), predictores_modelos[nombre])
                       for nombre in reajustes}, names=["modelo", "variable"]))


## Conclusión integrada · generada con los resultados del archivo

La siguiente celda redacta un resumen con los números calculados en esta ejecución. Separa la diferencia entre covarianza y correlación, la semidefinición positiva de las matrices, la redundancia entre contaminantes y la sensibilidad del ajuste a observaciones influyentes.

**Límites:** las correlaciones globales mezclan estaciones; la eliminación de faltantes puede seleccionar una muestra distinta; las observaciones horarias pueden depender entre sí; las métricas reportadas son de ajuste y no de pronóstico. Retirar puntos influyentes no resuelve automáticamente la colinealidad ni la dependencia temporal observada en el ejercicio anterior.


In [ ]:
texto_final = [
    f"En 2026 se organizó una tabla con {len(tabla):,} filas estación-hora y "
    f"{len(CONTAMINANTES)} contaminantes. En {ESTACION}, sobre "
    f"{int(resumen_b.loc[ESTACION, 'n_pares']):,} pares completos, la covarianza CO–NOX fue "
    f"{resumen_b.loc[ESTACION, 'covarianza']:.4f} y la correlación "
    f"{resumen_b.loc[ESTACION, 'correlacion']:.4f}; la primera depende de las unidades "
    "y la segunda es una medida estandarizada.",
    mensaje_c + f" El eigenvalor mínimo fue {eig_pares.min():.6g} por pares y "
    f"{eig_completos.min():.6g} en casos completos.",
    f"Para PM2.5 en {ESTACION}, ambos modelos usaron {len(muestra):,} horas: "
    f"el R² con todos los contaminantes fue {modelos['Todos'].rsquared:.4f} "
    f"y con gases fue {modelos['Gases'].rsquared:.4f}. La igualdad "
    f"PM10 = PM2.5 + PMCO se cumplió exactamente en "
    f"{identidades['PM10 - PM2.5 - PMCO'].eq(0).mean():.1%} de esas horas. "
    "Esto debe considerarse antes de interpretar un R² alto como capacidad predictiva."
]
for nombre in modelos:
    row = resumen_e.loc[nombre]
    texto_final.append(
        f"Para {nombre}, la matriz H semanal resultó idempotente dentro de tolerancia numérica, "
        f"con traza {row['traza_H']:.4f}, rango {int(row['rango_X'])} y "
        f"{int(row['p_columnas'])} columnas incluyendo intercepto. "
        "El leverage del periodo completo se calculó sin construir H."
    )
for item in comparacion_ranking:
    nombre = item['modelo']
    row = resumen_f.loc[nombre]
    texto_final.append(
        f"En {nombre}, coincidieron {item['coincidencias_entre_top10']} de las diez "
        "observaciones de mayor Cook con las de mayor leverage. Al retirar sus diez "
        f"mayores Cook, el R² del reajuste fue {row['R2_reajuste_en_retenidas']:.4f}. "
        f"En esas mismas filas retenidas, el RMSE pasó de {row['RMSE_original_en_retenidas']:.4f} "
        f"a {row['RMSE_reajuste_en_retenidas']:.4f}. "
        "La comparación describe sensibilidad dentro de muestra, no mejora predictiva demostrada."
    )
for nombre in set(modelos) - set(influencia):
    texto_final.append(f"En {nombre}, Cook y el reajuste por Cook no se interpretaron porque el ajuste fue numéricamente perfecto.")

display(Markdown("### Conclusión\n\n" + "\n\n".join(texto_final)))
# Variable disponible para copiar el texto al reporte:
conclusion_markdown = "\n\n".join(texto_final)
